In [1]:
#Import the necessary libraries and load environment variables
import os

from anthropic import Anthropic
from dotenv import load_dotenv
from policy import POLICY
import sqlite3
import requests

load_dotenv()

client = Anthropic()


python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 9
python-dotenv could not parse statement starting at line 10


In [2]:
import json
from operator import mod
import uuid
import sqlite3
from datetime import datetime, timezone
from typing import Dict, Tuple

'''
Make changes as per retry_policy_v1.json structure. 
The current implementation assumes a specific structure for the retry policy JSON file. 
If the structure of retry_policy_v1.json is different, you may need to adjust the code accordingly. 
For example, if the error classifications are nested differently or if there are additional fields, 
you will need to modify the classify_error and should_retry methods to correctly parse and utilize 
the information from the JSON file.
'''
class ClassifierMapping:
    def __init__(self, mapping_file="classifier_mapping.json"):
        with open(mapping_file) as f:
            self.mapping = json.load(f)
            self.by_exception = self.mapping.get("by_exception", {})
            self.by_status = self.mapping.get("by_status", {})
            self.status_ranges = self.mapping.get("status_ranges", [])

    def _names(self, cls) -> list[str]:
        """Return list of names to check for a class."""
        mod, qn = cls.__module__, cls.__qualname__
        root = mod.split(".")[0]
        return [f"{mod}.{qn}", f"{root}.{qn}", qn]   # full → root package → bare
    
    def get_error_category(self, exc: Exception | None=None, response=None) -> tuple[str, bool]:
        """Returns (category, was_default). Exception path wins over status path."""
        if exc is not None:
            for cls in type(exc).__mro__:
                for name in self._names(cls):
                    if name in self.by_exception:
                        return self.by_exception[name], False
            return "unknown", True

        if response is not None and response.status_code >= 400:
            code = str(response.status_code)
            if code in self.by_status:
                return self.by_status[code], False
            for r in self.status_ranges:
                if r["min"] <= response.status_code <= r["max"]:
                    return r["category"], False
            return "unknown", True

        return "unknown", True

class RetryPolicy:
    def __init__(self, policy_file="retry_policy_v1.json"):
        with open(policy_file) as f:
            self.config = json.load(f)
        self.policy_version = self.config["policy_version"]
    
    def classify_error(self, error_category: str) -> Dict:
        """Classify error, fallback to default if unknown."""
        if error_category in self.config["error_classifications"]:
            return self.config["error_classifications"][error_category], False
        return self.config["default_classification"], True
    
    def should_retry(self, error_category: str, current_attempt: int) -> Tuple[bool, str]:
        """Orchestrator asks: should I retry?"""
        classification, is_default = self.classify_error(error_category)
        
        if not classification.get("is_retryable", False):
            return False, f"{error_category} is non-retryable"
        
        max_attempts = classification.get("max_attempts", 0)
        if current_attempt >= max_attempts:
            return False, f"Max attempts ({max_attempts}) exhausted"
        
        return True, classification.get("description", "Retrying per policy")

class AuditLogger:
    def __init__(self, db_path="orchestrator.db"):
        self.conn = sqlite3.connect(db_path)
        self.create_tables()
    
    def create_tables(self):
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS events (
            event_id TEXT PRIMARY KEY,
            correlation_id TEXT NOT NULL,
            created_at TEXT NOT NULL,
            event_type TEXT NOT NULL,
            
            tool_name TEXT,
            tool_input TEXT,
            tool_output TEXT,
            tool_use_id TEXT,
            
            error_category TEXT,
            error_type TEXT,
            error_message TEXT,
            is_retryable BOOLEAN,
            was_classified_as_default BOOLEAN,
            
            error_source_event_id TEXT,
            retry_decision_source_event_id TEXT,
            decision TEXT,
            reason TEXT,
            retry_budget_remaining INTEGER,
            retry_policy_version TEXT,
            attempt_number INTEGER,
            
            final_answer_context TEXT,
            context_source_event_id TEXT,
            final_answer_text TEXT,
            
            model_version TEXT,
            user_id TEXT,
            
            UNIQUE(correlation_id, tool_use_id) WHERE tool_use_id IS NOT NULL,
            FOREIGN KEY(error_source_event_id) REFERENCES events(event_id),
            FOREIGN KEY(retry_decision_source_event_id) REFERENCES events(event_id),
            FOREIGN KEY(context_source_event_id) REFERENCES events(event_id)
        );
        
        CREATE INDEX IF NOT EXISTS idx_correlation_time 
        ON events(correlation_id, created_at);
        """)
        self.conn.commit()
    
    def log_event(self, event_data: Dict):
        """Log event. Idempotent on event_id."""
        try:
            self.conn.execute("""
            INSERT INTO events VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
            """, (
                event_data.get("event_id"),
                event_data.get("correlation_id"),
                event_data.get("created_at"),
                event_data.get("event_type"),
                event_data.get("tool_name"),
                json.dumps(event_data.get("tool_input")) if event_data.get("tool_input") else None,
                json.dumps(event_data.get("tool_output")) if event_data.get("tool_output") else None,
                event_data.get("tool_use_id"),
                event_data.get("error_category"),
                event_data.get("error_type"),
                event_data.get("error_message"),
                event_data.get("is_retryable"),
                event_data.get("was_classified_as_default"),
                event_data.get("error_source_event_id"),
                event_data.get("retry_decision_source_event_id"),
                event_data.get("decision"),
                event_data.get("reason"),
                event_data.get("retry_budget_remaining"),
                event_data.get("retry_policy_version"),
                event_data.get("attempt_number"),
                event_data.get("final_answer_context"),
                event_data.get("context_source_event_id"),
                event_data.get("final_answer_text"),
                event_data.get("model_version"),
                event_data.get("user_id"),
            ))
            self.conn.commit()
        except sqlite3.IntegrityError as e:
            if "UNIQUE" in str(e):
                print(f"Event {event_data.get('event_id')} already logged")
            else:
                raise
    
    def create_event(self, event_type: str, **kwargs) -> Dict:
        """Factory for creating events."""
        return {
            "event_id": str(uuid.uuid4()),
            "created_at": datetime.now(timezone.utc).isoformat(),
            "event_type": event_type,
            **kwargs
        }
    
    def get_timeline(self, correlation_id: str):
        """Retrieve full timeline for a conversation."""
        cursor = self.conn.execute("""
        SELECT event_id, created_at, event_type, tool_name, error_category, 
               decision, attempt_number, final_answer_context
        FROM events
        WHERE correlation_id = ?
        ORDER BY created_at
        """, (correlation_id,))
        return cursor.fetchall()

In [3]:
#Create a function to add user messages to the conversation
#Create a function to add assistant messages to the conversation
#Create a function to converse
from typing import Optional


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})

def converse(messages, model = "claude-haiku-4-5-20251001", max_tokens_to_sample=300,
             temperature: Optional[float] = None,
             stop_sequences: Optional[list[str]] = None,
             tools: Optional[list[dict]] = None,
             system_prompt: Optional[str] = None) -> dict:

    arg_list = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens_to_sample
    }

    if temperature is not None:
        arg_list["temperature"] = temperature

    if stop_sequences is not None:
        arg_list["stop_sequences"] = stop_sequences

    if tools is not None:
        arg_list["tools"] = tools

    if system_prompt is not None:
        arg_list["system_prompt"] = system_prompt

    response = client.messages.create(
        **arg_list
    )

    return response



In [4]:
# Tool and Schema Definitions

# Tool get_weather with Open-Meteo API
def get_weather(latitude: float, longitude: float, timeout_s: float = 10.0) -> requests.Response:
    """
    Get the current weather for a given latitude and longitude.
    """
    import requests

    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"
    response = requests.get(url, timeout=timeout_s)
    return response


def get_customer(customer_id: str):
    """
    Get a customer by ID.
    """
    return {
        "id": customer_id,
        "name": "John Doe",
        "email": "john.doe@example.com",
        "acc_type": "business"
    }

def get_business_details(customer_id: str):
    """
    Get business details for a customer.
    """
    return {
        "customer_id": customer_id,
        "business_name": "Doe Enterprises",
        "industry": "Retail",
        "location": "New York, NY"
    }

get_weather_schema = {
    "name": "get_weather",
    "description": "Get the current weather for a given latitude and longitude.",
    "input_schema": {
        "type": "object",
        "properties": {
            "latitude": {
                "type": "number",
                "description": "Latitude of the location"
            },
            "longitude": {
                "type": "number",
                "description": "Longitude of the location"
            }
        },
        "required": []
    }
}

get_customer_schema = {
    "name": "get_customer",
    "description": "Get customer object based on customer ID. It returns property of a customer in json format.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "Customer ID to lookup"
            }
        },
        "required": []
    }
}

get_business_details_schema = {
    "name": "get_business_details",
    "description": "Get customer's business details based on customer ID. It returns property of a customer's business in json format. Automatically call this tool whenever a customer's account_type is 'business', without asking the user first.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "Customer ID's business to lookup"
            }
        },
        "required": []
    }
}

In [5]:
def execute_tool(tool_name: str, tool_inputs: dict, timeout_s: float = 10.0) -> dict:
    ADVISORY_RETRYABLE = {"timeout", "network", "upstream_5xx", "rate_limit",
                      "upstream_contract_violation"}
    error_classifier = ClassifierMapping()
    if tool_name != "get_weather":
        return {
            "ok": False,
            "errorCategory": "internal_error",
            "message": f"Unknown tool: {tool_name}",
            "attemptedInput": tool_inputs,
    
        }

    # Validate BEFORE the call — a missing key is our bug, not the API's.
    try:
        lat = float(tool_inputs["latitude"])
        lon = float(tool_inputs["longitude"])
    except (KeyError, TypeError, ValueError) as e:
        return {
            "ok": False,
            "errorCategory": "internal_error",
            "message": f"Tool input contract violation: {e}",
            "attemptedInput": tool_inputs,
        }

    try:
        response = get_weather(lat, lon, timeout_s=timeout_s)
    except Exception as e:
        category, was_default = error_classifier.get_error_category(exc=e)
        return {
            "ok": False,
            "errorCategory": category,
            "wasClassifiedAsDefault": was_default,
            "toolReportedRetryable": category in ADVISORY_RETRYABLE,
            "attemptedInput": {"latitude": lat, "longitude": lon},
            "message": f"{type(e).__name__}: {e}",
            "exceptionType": type(e).__name__
        }

    if response.status_code >= 400:
        category, was_default = error_classifier.get_error_category(response=response)
        try:
            upstream_reason = response.json().get("reason")
        except Exception:
            upstream_reason = response.text[:200]
        return {
            "ok": False,
            "errorCategory": category,
            "wasClassifiedAsDefault": was_default,
            "toolReportedRetryable": category in ADVISORY_RETRYABLE,
            "attemptedInput": {"latitude": lat, "longitude": lon},
            "message": upstream_reason or f"HTTP {response.status_code}",
            "httpStatusCode": response.status_code,
            "retryAfter": response.headers.get("Retry-After"),
        }

    try:
        return {"ok": True, "data": response.json()}
    except Exception as e:
        return {
            "ok": False,
            "errorCategory": "upstream_contract_violation",
            "attemptedInput": {"latitude": lat, "longitude": lon},
            "message": f"200 with unparseable body: {e}",
            "httpStatusCode": response.status_code,
        }

# This method needs to return a list of results, one per tool actually requested, each tagged with its own tool_use_id
def execute_tools(request_id, parsed_response):
    if parsed_response and parsed_response["tools"]:
        tools = parsed_response["tools"]
        results = []
        for tool in tools:
            tool_name = tool["name"]
            tool_inputs = tool["inputs"]
            tool_id = tool["tool_id"]
            ifPassed, reason = check_prerequisites(request_id, tool_name, tool_inputs)
            if ifPassed:
                result = execute_tool(tool_name, tool_inputs)
                # persist_tool_call(request_id, tool_name, tool_inputs, result)
            else:
                result = {"error": reason}
            results.append({"tool_use_id": tool_id, "result": result})
        return results

In [47]:
for label, args in [
    ("valid",        {"latitude": -37.81, "longitude": 144.96}),
    ("bad_coords",   {"latitude": 999, "longitude": 999}),
    ("wrong_keys",   {"lat": -37.81, "lon": 144.96}),
    ("timeout",      {"latitude": -37.81, "longitude": 144.96}),
]:
    r = execute_tool("get_weather", args, timeout_s=0.001 if label == "timeout" else 10.0)
    print(f"OUTPUT: {label}: {r.get('ok')}, {r.get('errorCategory')}, {r.get('httpStatusCode')}")

OUTPUT: valid: True, None, None
OUTPUT: bad_coords: False, invalid_input, 400
OUTPUT: wrong_keys: False, internal_error, None
OUTPUT: timeout: False, timeout, None


In [6]:
try:
    get_weather(-37.81, 144.96, timeout_s=0.001)
except Exception as e:
    print(type(e), [f"{c.__module__}.{c.__qualname__}" for c in type(e).__mro__])

<class 'requests.exceptions.ConnectTimeout'> ['requests.exceptions.ConnectTimeout', 'requests.exceptions.ConnectionError', 'requests.exceptions.Timeout', 'requests.exceptions.RequestException', 'builtins.OSError', 'builtins.Exception', 'builtins.BaseException', 'builtins.object']


In [7]:
#Write a method to parse assistance message
from anthropic.types import Message, ToolUseBlock


def parse_response(response) -> dict:
    json_response = {"text": "", "tools": []}
    
    if response and isinstance(response, Message):
        
        contents = response.content
        for content in contents:
            if isinstance(content, ToolUseBlock):
                json_response["tools"].append({
                    "inputs": content.input,
                    "name": content.name,
                    "tool_id": content.id
                })
            elif isinstance(content, str):
                json_response["text"] += content
            else:
                json_response["text"] += str(content)
    return json_response

In [8]:
import random

def parse_retry_after(raw) -> int | None:
    if raw is None:
        return None
    try:
        return int(float(raw)) * 1000
    except (TypeError, ValueError):
        return None          # HTTP-date form; ignore for this exercise

def compute_backoff(backoff: dict, attempt: int, retry_after_raw=None) -> tuple[int, bool]:
    """Sleep duration in ms for a 1-based retry attempt.
    Returns (sleep_ms, must_escalate)."""
    strategy = backoff["strategy"]
    base_ms  = backoff["base_ms"]
    max_ms   = backoff.get("max_ms", base_ms)

    if strategy == "fixed":
        raw_ms = base_ms
    elif strategy == "exponential":
        raw_ms = base_ms * (backoff.get("multiplier", 2) ** (attempt - 1))
    else:
        raise ValueError(f"Unknown backoff strategy: {strategy!r}")

    capped = min(raw_ms, max_ms)

    if backoff.get("jitter") == "full":
        sleep_ms = random.uniform(0, capped)
    else:
        sleep_ms = capped

    if backoff.get("honor_retry_after"):
        ra_ms = parse_retry_after(retry_after_raw)
        if ra_ms is not None:
            if ra_ms > max_ms and backoff.get("on_retry_after_exceeds_max") == "escalate":
                return 0, True
            sleep_ms = max(sleep_ms, ra_ms) + random.uniform(0, 1000)  # jitter upward only

    return int(sleep_ms), False

In [ ]:

import time


def initialize_orchestrator():
    global audit, retry_policy, error_classifier
    # In your Exercise 4 orchestrator:
    audit = AuditLogger()
    retry_policy = RetryPolicy()


# ... orchestrator loop ...
initialize_orchestrator()
messages  = []
user_query = "Get weather for lat=35.6895, lon=139.6917 (Tokyo)"
add_user_message(messages, f"{user_query}")
correlation_id = str(uuid.uuid4())
retry_count_by_category = {}

# Log user query
# The event currently does not include the actual user query text in the event. If you want to log the user query text, you can add it to the event data.
user_query_event = audit.create_event(
    "user_query",
    correlation_id=correlation_id,
    user_query_text=user_query,
    model_version="claude-haiku-4-5"
)
audit.log_event(user_query_event)

response = converse(messages, tools=[get_weather_schema, get_customer_schema, get_business_details_schema])
MAX_ITERATIONS = 5
iteration_count = 0
escalated = False



while (response.stop_reason == "tool_use") and (iteration_count < MAX_ITERATIONS):
    iteration_count += 1
    # Keep conversing until the model stops requesting tools
    add_assistant_message(messages, response.content)
    # response = converse(messages, tools=[get_customer_schema, get_business_details_schema])
    parsed_response = parse_response(response)
    # check if there is any tool use in the parsed response
    # Tool_response returns a list of result, one per tool actually requested, each tagged with its own tool_use_id
    tool_response = execute_tools(correlation_id, parsed_response)

    # for each tool in the response create user message with the tool result and add it to the messages list
    user_message_content = []
    for result in tool_response:
        if result["result"].get("ok"):
            user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_result", "content": [{"type": "text", "text": json.dumps(result["result"]["data"])}]})
        else:
            # Log Error Event with tool reported retryable, policy handling and classification conflict
            # If should retry then log the decision and branch on handling
            audit_event = audit.create_event(
                "error",
                correlation_id=correlation_id,
                error_category=result["result"].get("errorCategory"),
                error_type="tool_error",
                error_message=result["result"].get("message"),
                is_retryable=result["result"].get("toolReportedRetryable"),
                was_classified_as_default=result["result"].get("wasClassifiedAsDefault"),
                model_version="claude-haiku-4-5"
            )
            audit.log_event(audit_event)

            handling = retry_policy.config["error_classifications"].get(result["result"].get("errorCategory"), {}).get("handling")

            if handling == "escalate":
                # Escalate to human, log final answer event with escalated context
                # Log tool result with error message and tool use id
                escalated = True
                last_error_event_id = audit_event["event_id"]
                user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_error", "content": [{"type": "text", "text": f"Tool {result['tool_use_id']} failed with error: {result['result'].get('message')}. Escalating to human as per policy."}]})

                final_answer_event = audit.create_event(
                    "final_answer",
                    correlation_id=correlation_id,
                    final_answer_context="escalated",
                    context_source_event_id=audit_event["event_id"],
                    final_answer_text="Escalated to human due to non-retryable error.",
                    model_version="claude-haiku-4-5"
                )
                audit.log_event(final_answer_event)

                break

            if handling == "return_to_model":
                # Feed error to model, loop continues
                user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_error", "content": [{"type": "text", "text": f"Tool {result['tool_use_id']} failed with error: {result['result'].get('message')}. Returning to model as per policy."}]})
                continue

            if handling == "orchestrator_retry":
                # If there is no conflict with policy handling, then classify and decide
                classification, is_default = retry_policy.classify_error(result["result"].get("errorCategory"))
                current_attempt = retry_count_by_category.get(result["result"].get("errorCategory"), 0)
                # Put it in the retry_count_by_category dict to keep track of the number of attempts for each error category
                retry_count_by_category[result["result"].get("errorCategory")] = current_attempt
                while current_attempt < classification["max_attempts"]:
                    # Keep auditing and retrying tool calls until the max attempts is reached or the tool call succeeds
                    should_retry, reason = retry_policy.should_retry(result["result"].get("errorCategory"), current_attempt)
                    # As long as orchetrator_retry, audit the decision with any conflict with should_retry from tool
                    # Then call the tool again with the same inputs and timeout, and log the result

                    retry_decision_event = audit.create_event(
                        "retry_decision",
                        correlation_id=correlation_id,
                        error_source_event_id=audit_event["event_id"],
                        error_category=result["result"].get("errorCategory"),
                        decision="retry",
                        reason=reason,
                        was_classified_as_default=is_default,
                        retry_policy_version=retry_policy.policy_version,
                        retry_budget_remaining=classification["max_attempts"] - current_attempt,
                        model_version="claude-haiku-4-5"
                    )
                    audit.log_event(retry_decision_event)
                    #Ececute the tool again with the same inputs and timeout
                    retry_result = execute_tool(result["result"].get("tool_name"), result["result"].get("attemptedInput"))
                    current_attempt += 1
                    retry_count_by_category[result["result"].get("errorCategory")] = current_attempt
                    # time.sleep(compute_backoff(classification["backoff"], current_attempt, result["result"].get("retryAfter"))/1000)
                    if retry_result.get("ok"):
                        user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_result", "content": [{"type": "text", "text": json.dumps(retry_result["data"])}]})
                        break
                    else:
                        continue
                # append the user message after retry exhaustion or success
                user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_error", "content": [{"type": "text", "text": f"Tool {result['tool_use_id']} failed with error: {result['result'].get('message')}. Retry attempts exhausted as per policy."}]})
            if handling is None:
                # If there is no handling defined in the policy, then escalate to human
                user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_error", "content": [{"type": "text", "text": f"Tool {result['tool_use_id']} failed with error: {result['result'].get('message')}. Escalating to human as per policy."}]})

                final_answer_event = audit.create_event(
                    "final_answer",
                    correlation_id=correlation_id,
                    final_answer_context="escalated",
                    context_source_event_id=audit_event["event_id"],
                    final_answer_text="Escalated to human due to non-retryable error.",
                    model_version="claude-haiku-4-5"
                )
                audit.log_event(final_answer_event)
                break
    add_user_message(messages, user_message_content)
    if escalated:
        break
    response = converse(messages, tools=[get_business_details_schema, get_customer_schema, get_weather_schema])



# At the end:
final_answer_event = audit.create_event(
    "final_answer",
    correlation_id=correlation_id,
    final_answer_context="normal" if not escalated else "escalated",
    context_source_event_id=last_error_event_id if escalated else None,
    final_answer_text="...",
    model_version="claude-haiku-4-5"
)
audit.log_event(final_answer_event)

# Verify timeline:
timeline = audit.get_timeline(correlation_id)
for row in timeline:
    print(row)

NameError: name 'audit' is not defined

In [ ]:



def query_history(correlation_id, requires):
    #Use sqlite server to query for correlation_id and customer_id for previous calls to the tool specified in requires. Return a list of tuples containing the input and output of each call.
    sqlite_conn = sqlite3.connect("tools.db")
    cursor = sqlite_conn.cursor()
    cursor.execute(
        "SELECT input, output FROM tool_calls WHERE correlation_id = ? AND tool_name = ?",
        (str(correlation_id), requires)
    )
    results = cursor.fetchall()
    sqlite_conn.close()
    return [(json.loads(inp), json.loads(out)) for inp, out in results]

def persist_tool_call(correlation_id, tool_name, tool_input, tool_output):
    conn = sqlite3.connect("tools.db")
    conn.execute(
        "INSERT INTO tool_calls (correlation_id, tool_name, input, output) VALUES (?, ?, ?, ?)",
        (str(correlation_id), tool_name, json.dumps(tool_input), json.dumps(tool_output)),
    )
    conn.commit()
    conn.close()

def persist_policy_rejection(correlation_id, tool_name, reason):
    conn = sqlite3.connect("tools.db")
    conn.execute(
        "INSERT INTO policy_rejections (correlation_id, tool_name, rejection_reason) VALUES (?, ?, ?)",
        (str(correlation_id), tool_name, reason),
    )
    conn.commit()
    conn.close()

def check_prerequisites(correlation_id, tool_name, current_input):
    for req in POLICY.get(tool_name, []):
        matches = query_history(correlation_id, req["requires"])
        print(f"Checking prerequisites for {tool_name}: {req['requires']}, found matches: {matches}")
        satisfied = any(
            all(prior_input.get(p) == current_input.get(c) for p, c in req["on"])
            and req["condition"](prior_result)
            for prior_input, prior_result in matches
        )
        if not satisfied:
            reason = f"Missing prerequisite: {req['requires']} for {tool_name}"
            # Persist the policy rejection in the database
            persist_policy_rejection(correlation_id, tool_name, reason)
            return False, reason
    return True, None



In [25]:
import sqlite3
conn = sqlite3.connect("tools.db")
print(conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())
conn.close()

[('tool_calls',), ('sqlite_sequence',), ('policy_rejections',)]


In [29]:
persist_policy_rejection("test-correlation", "get_business_details", "test reason")
conn = sqlite3.connect("tools.db")
print(conn.execute("SELECT * FROM tool_calls").fetchall())
conn.close()

[(67, '8c2bace9-cb00-42c0-82f3-2efb0012fc47', 'get_customer', '{"customer_id": "9956"}', '{"id": "9956", "name": "John Doe", "email": "john.doe@example.com", "acc_type": "business"}'), (68, '8c2bace9-cb00-42c0-82f3-2efb0012fc47', 'get_business_details', '{"customer_id": "9956"}', '{"customer_id": "9956", "business_name": "Doe Enterprises", "industry": "Retail", "location": "New York, NY"}'), (69, '4d3e281a-0a7d-413f-908a-355b73ef71dc', 'get_customer', '{"customer_id": "3232"}', '{"id": "3232", "name": "John Doe", "email": "john.doe@example.com", "acc_type": "business"}'), (70, '4d3e281a-0a7d-413f-908a-355b73ef71dc', 'get_business_details', '{"customer_id": "3232"}', '{"customer_id": "3232", "business_name": "Doe Enterprises", "industry": "Retail", "location": "New York, NY"}'), (71, '99fd6e17-c0f6-40e5-a309-01c5ffbd77fa', 'get_customer', '{"customer_id": "5688"}', '{"id": "5688", "name": "John Doe", "email": "john.doe@example.com", "acc_type": "business"}'), (72, '99fd6e17-c0f6-40e5-a3

In [7]:
"""
import uuid
messages  = []
add_user_message(messages, "Get business details for customer ID 1234")
correlation_id = uuid.uuid4()

response = converse(messages, tools=[get_customer_schema, get_business_details_schema])
add_assistant_message(messages, response.content)
parsed_response = parse_response(response)
print(parsed_response)
print(messages)
tmp = messages
"""

'\nimport uuid\nmessages  = []\nadd_user_message(messages, "Get business details for customer ID 1234")\ncorrelation_id = uuid.uuid4()\n\nresponse = converse(messages, tools=[get_customer_schema, get_business_details_schema])\nadd_assistant_message(messages, response.content)\nparsed_response = parse_response(response)\nprint(parsed_response)\nprint(messages)\ntmp = messages\n'

In [8]:
"""
tool_response = execute_tools(parsed_response)
print(tool_response)
"""

'\ntool_response = execute_tools(parsed_response)\nprint(tool_response)\n'

In [9]:
"""import json


messages = tmp
user_message_content = [{"tool_use_id": parsed_response["tool_id"], "type": "tool_result", "content": [{"type": "text", "text": json.dumps(tool_response)}]}]
add_user_message(tmp, user_message_content)
tmp
"""


'import json\n\n\nmessages = tmp\nuser_message_content = [{"tool_use_id": parsed_response["tool_id"], "type": "tool_result", "content": [{"type": "text", "text": json.dumps(tool_response)}]}]\nadd_user_message(tmp, user_message_content)\ntmp\n'

In [10]:
"""
print(messages)
resp = converse(messages, tools=[get_business_details_schema, get_customer_schema])
"""

'\nprint(messages)\nresp = converse(messages, tools=[get_business_details_schema, get_customer_schema])\n'

In [11]:
import policy
import importlib
importlib.reload(policy)
print(policy.POLICY)

{'get_business_details': [{'requires': 'get_customer', 'on': [('customer_id', 'customer_id')], 'condition': <function <lambda> at 0x797f89057420>}], 'get_customer': []}


In [28]:
import sqlite3
conn = sqlite3.connect("tools.db")
#conn.execute("DELETE FROM tool_calls")
# Select from policy_rejections table to see if any policy rejections have been logged  
cursor = conn.execute(""" 
SELECT * FROM policy_rejections;
""")

results = cursor.fetchall()
for row in results:
    print(row)

conn.commit()
conn.close()

(1, '8c2bace9-cb00-42c0-82f3-2efb0012fc47', 'get_business_details', '2026-09-05T04:37:00.636', 'Missing prerequisite: get_customer for get_business_details')
(2, '3e4cf0d5-c5e5-4bdd-a46a-0e3883277746', 'get_business_details', '2026-09-05T04:37:14.992', 'Missing prerequisite: get_customer for get_business_details')
(3, '7a159e60-bfe0-4926-82da-ec85dd633ee8', 'get_business_details', '2026-09-05T04:37:26.113', 'Missing prerequisite: get_customer for get_business_details')
(4, '45f0d8be-d5a6-4ba6-80ed-b78644de780b', 'get_business_details', '2026-09-05T04:37:31.296', 'Missing prerequisite: get_customer for get_business_details')
(5, '53b2a2e6-fee5-470f-8ab5-8f1ed478d63d', 'get_business_details', '2026-09-05T04:37:36.094', 'Missing prerequisite: get_customer for get_business_details')
(6, '895a05d1-2953-4402-8422-24722700c012', 'get_business_details', '2026-09-05T04:37:43.630', 'Missing prerequisite: get_customer for get_business_details')
(7, '20906de9-e7c5-4336-ab17-37001b706c3b', 'get_bus

In [ ]:
# persist_tool_call("test-uuid", "get_customer", {"customer_id": "1234"}, {"acc_type": "business"})
# print(query_history("test-uuid", "get_customer"))

[({'customer_id': '1234'}, {'acc_type': 'business'})]


In [ ]:
# check_prerequisites("test-uuid", "get_business_details", {"customer_id": "1234"})

Checking prerequisites for get_business_details: get_customer, found matches: [({'customer_id': '1234'}, {'acc_type': 'business'})]


(True, None)

In [16]:
import sqlite3
conn = sqlite3.connect("tools.db")
cursor = conn.cursor()
"""
cursor.execute(\"""
    SELECT correlation_id, tool_name, input FROM tool_calls
    WHERE correlation_id IN ('d1467011-8f26-4f19-8d29-5519e04f3edf', '7bb48b1f-43ab-485e-91c7-4adfe11e963e')
    ORDER BY correlation_id, id
\""")
"""
cursor.execute("""
    DELETE FROM tool_calls;
""")
#for row in cursor.fetchall():
#    print(row)
conn.close()